# ⚡ ClipStack — Free Cloud GPU Engine (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openclaw-by-hiten/Clipstack/blob/main/ClipStack_Cloud_Engine.ipynb)
[![Live Web Studio](https://img.shields.io/badge/Live%20Studio-GitHub%20Pages-yellow?style=flat&logo=github)](https://openclaw-by-hiten.github.io/Clipstack/)

Run the entire ClipStack AI video clipping studio on Google's **free NVIDIA T4 GPU** and **100 GB cloud disk**.
- **0 MB** disk space used on your personal computer.
- **0%** local CPU/GPU load.
- Full AI video downloading, Whisper word-level transcription, and 9:16 vertical rendering in the cloud.

---

### 🚀 Quick Start (3 Steps):
1. **Runtime Check**: Click **Runtime** > **Change runtime type** in top menu and select **T4 GPU** (Free).
2. **Run All Cells**: Press **Runtime** > **Run all** (or `Ctrl + F9`).
3. **Connect to Web App**: Copy the generated `https://xxxx.trycloudflare.com` URL from Cell 3, open [https://openclaw-by-hiten.github.io/Clipstack/](https://openclaw-by-hiten.github.io/Clipstack/), click **Backend** in the top right, and paste it!


In [ ]:
# Step 1: Verify NVIDIA GPU & Install System Dependencies
!nvidia-smi

print('\n📦 Installing system dependencies (FFmpeg, audio codecs, DejaVu fonts)...')
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 fonts-dejavu-core msttcorefonts > /dev/null 2>&1
print('✅ System dependencies installed successfully!')


In [ ]:
# Step 2: Clone or Update ClipStack Repository & Install Python Packages
import os

REPO_PATH = '/content/Clipstack'

if not os.path.exists(REPO_PATH):
    print('📥 Cloning ClipStack from GitHub...')
    !git clone https://github.com/openclaw-by-hiten/Clipstack.git $REPO_PATH
else:
    print('🔄 Pulling latest updates from GitHub...')
    !cd $REPO_PATH && git pull

%cd $REPO_PATH

print('\n📦 Installing Python requirements (FastAPI, Faster-Whisper, yt-dlp, etc.)...')
!pip install -q -r requirements.txt
print('✅ Python environment ready!')


In [ ]:
# Step 3: Start ClipStack FastAPI Server & Launch Cloudflare Tunnel
import os
import time
import subprocess
import urllib.request
import re
from IPython.display import display, HTML

# 1. Download official Cloudflare Tunnel binary for Linux amd64
cloudflared_path = '/usr/local/bin/cloudflared'
if not os.path.exists(cloudflared_path):
    print('⬇️ Downloading Cloudflare Tunnel (cloudflared)...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_path
    )
    os.chmod(cloudflared_path, 0o755)
    print('✅ cloudflared installed successfully!')

# 2. Terminate any previous instances
!pkill -f uvicorn > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# 3. Launch ClipStack FastAPI backend server
print('🚀 Launching ClipStack FastAPI Backend on port 8000...')
server_log = open('/content/server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    cwd='/content/Clipstack'
)
time.sleep(4)

# 4. Start Cloudflare Tunnel pointing to port 8000
print('🌐 Creating secure HTTPS tunnel via Cloudflare...')
tunnel_log = open('/content/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 5. Extract public trycloudflare.com URL
public_url = None
start_time = time.time()
while time.time() - start_time < 35:
    line = tunnel_proc.stdout.readline()
    if line:
        tunnel_log.write(line)
        tunnel_log.flush()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print('\n' + '=' * 65)
    print(f'🎉 CLIPSTACK CLOUD GPU ENGINE IS ONLINE!')
    print(f'🔗 Public Backend URL: {public_url}')
    print('=' * 65 + '\n')
    display(HTML(f'''
    <div style="background: linear-gradient(135deg, #0f172a, #1e1b4b); border: 2px solid #6366f1; border-radius: 14px; padding: 24px; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; color: #f8fafc; max-width: 650px;">
        <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 12px;">
            <span style="font-size: 24px;">⚡</span>
            <h2 style="margin: 0; color: #38bdf8; font-size: 20px; font-weight: 700;">ClipStack Cloud Engine is Active!</h2>
        </div>
        <p style="margin: 0 0 16px 0; color: #94a3b8; font-size: 14px; line-height: 1.5;">
            Copy your live Cloud backend URL below and connect it to your live GitHub Pages site:
        </p>
        <div style="background: #020617; border: 1px dashed #38bdf8; border-radius: 8px; padding: 12px 16px; margin-bottom: 18px; word-break: break-all;">
            <span style="color: #4ade80; font-family: monospace; font-size: 16px; font-weight: bold;">{public_url}</span>
        </div>
        <div style="display: flex; flex-direction: column; gap: 8px; font-size: 13px; color: #cbd5e1;">
            <div><b>1. Open your Web App:</b> <a href="https://openclaw-by-hiten.github.io/Clipstack/" target="_blank" style="color: #60a5fa; text-decoration: underline; font-weight: 600;">https://openclaw-by-hiten.github.io/Clipstack/</a></div>
            <div><b>2. Click:</b> <span style="background: #334155; padding: 2px 6px; border-radius: 4px; font-family: monospace;">Backend</span> (top right corner)</div>
            <div><b>3. Paste:</b> Enter the URL above and click <b>Connect</b>!</div>
        </div>
    </div>
    '''))
else:
    print('⚠️ Could not automatically detect tunnel URL in 35s. Please check /content/tunnel.log for details.')

# Stream tunnel output to keep cell alive
try:
    for line in tunnel_proc.stdout:
        tunnel_log.write(line)
        tunnel_log.flush()
except KeyboardInterrupt:
    print('🛑 Stopping server and tunnel...')
    server_proc.terminate()
    tunnel_proc.terminate()
